In [18]:
from http.client import responses

from anyio.lowlevel import checkpoint
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated,Literal
from pydantic import Field, BaseModel
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver


In [19]:
class ChatState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]

In [20]:
load_dotenv()

# Model
model = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [21]:
def chat_node(state: ChatState) -> ChatState:
    # take user query state
    message = state['message']
    # send to llm
    response = model.invoke(message)

    #response store state
    return {'message': [response]}


In [22]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

# Node
graph.add_node('chat_node', chat_node)

# Edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [23]:
workflow = graph.compile(checkpointer=checkpointer)

In [25]:
# initial_state = {
#     'message': [HumanMessage(content='what is capital of india')]
# }
# result = workflow.invoke(initial_state)
#
# print(result['message'][-1].content[0]['text'])

In [26]:
thread_id = '1'

while True:
    user_message = input('Enter your message: ')
    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    int_msg = {
        'message': [HumanMessage(content=user_message)]
    }
    config = {'configurable': {'thread_id': thread_id}}
    response = workflow.invoke(int_msg, config=config)

    print('Bot:', response['message'][-1].content[0]['text'])



User: hi
Bot: Hello! How can I help you today?
User: I am manoj
Bot: Nice to meet you, Manoj! How can I help you today?
User: what is the capital of india
Bot: The capital of India is **New Delhi**.
User: what is my name

Bot: Your name is **Manoj**. You mentioned it just a moment ago!
User: I have 2 apple and 3 banana
Bot: Got it, Manoj! So you have a total of **5 pieces of fruit**.

Is there anything else you'd like to do with that information, or shall I keep track of your "inventory" for you?
User: what is capital of punjab
Bot: The capital of the Indian state of Punjab is **Chandigarh**. 

Interestingly, Chandigarh also serves as the capital of the neighboring state of Haryana and is a Union Territory itself. 

(If you were referring to the Punjab province in **Pakistan**, its capital is **Lahore**.)
User: how many fruits i have
Bot: You have a total of **5 fruits**:
*   2 apples
*   3 bananas
User: okay thanks bye
Bot: You're welcome, Manoj! You have your **5 fruits** (2 apples a

In [27]:
workflow.get_state(config=config)

StateSnapshot(values={'message': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='35686c95-2102-4498-beae-0830955940ee'), AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EqMCCqACAb4+9vu4R2ZJhcKlUCPSg0k0OvJUV/LDfs0wsN8dsoYnr1IwVb27E3naN8B4HQ0aKRAHRH33gzKe0ixfwTNV66jAwj4loBkz6CZqMjmdhnAH+G4/StdpkVl2WZ5J6A9NSs4Da/k/3vR2RseByCt5FsGTnsmsvh1ugd22DOgHx7gYoIB/wh2E0gnU7iM4S6uEyd6IluQ9LZ81u4xIgXTwDXdGQb7HqLu2FTlJQz2YBWs5FTkwUrrEG1r/v6VijLxivoE+YzuBTtazwsTud8xxgGAhySBtdMWU0K+EzO0JeiB4S/00bLtwvSda2+syJgSkwAVvj+V/dz1Fn/mdjPkIjl0eeG55tHFpHg+gMBz/ib7PG9ToaQHMhHUfaULXLUlZ'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d868d-1556-7ee0-a6ec-545e350fbf55-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 73, 'total_tokens': 75, 'input_token_detai